In [1]:
PAYLOAD_PATH = ""
TOKEN = ""
ID_FONTE = ""
MUNICIPIO = ""
SECRETARIA = ""
UNIDADE_ORGANIZACIONAL = ""

StatementMeta(, 1fcce476-d977-420d-bbb6-f8a609245845, 3, Finished, Available, Finished, False)

In [2]:
%run ./nb_utils_request_api

StatementMeta(, 1fcce476-d977-420d-bbb6-f8a609245845, 4, Finished, Available, Finished, True)

In [ ]:
METADADOS_FONTE = {
    "municipio": MUNICIPIO,
    "secretaria": SECRETARIA,
    "unidade_organizacional": UNIDADE_ORGANIZACIONAL,
}
COLS_FATO_SOLICITACOES = [
            "servico",
            "status_fluxo",
            "data_criacao",
            "data_finalizacao",
            "solicitante",
            "origem",
            "data_carga",
]


def gerar_fato_solicitacoes(df_solicitacoes, METADADOS_FONTE, COLS_FATO_SOLICITACOES):
    
    df_solicitacoes_fato = df_solicitacoes[["id_os"] + COLS_FATO_SOLICITACOES].copy()

    for coluna, valor in METADADOS_FONTE.items():
        df_solicitacoes_fato[coluna] = valor

    return df_solicitacoes_fato


def gerar_fato_campos(df_solicitacoes, METADADOS_FONTE, COLS_FATO_SOLICITACOES):
    cols_drop = [
        c
        for c in COLS_FATO_SOLICITACOES
        if c not in ("id_os", "origem", "data_carga", "servico")
    ]
    df_fato_campos = df_solicitacoes.drop(columns=cols_drop)
    df_fato_campos_melt = df_fato_campos.melt(
        id_vars=["id_os", "servico", "origem", "data_carga"],
        var_name="campo",
        value_name="valor",
    )
    df_fato_campos_melt["valor"] = np.where(
        df_fato_campos_melt["valor"] == "", np.nan, df_fato_campos_melt["valor"]
    )
    df_fato_campos_melt = df_fato_campos_melt.dropna(subset="valor").reset_index(
        drop=True
    )
    df_fato_campos_melt = df_fato_campos_melt[
        ["id_os", "servico", "campo", "valor", "origem", "data_carga"]
    ].copy()
    for coluna, valor in METADADOS_FONTE.items():
        df_fato_campos_melt[coluna] = valor

    return df_fato_campos_melt


def gerar_fato_etapas(df_etapas, METADADOS_FONTE):
    for coluna, valor in METADADOS_FONTE.items():
        df_etapas[coluna] = valor
    return df_etapas


def write_bronze(df_solicitacoes_fato, df_fato_campos_melt, df_etapas):

    tabelas = {
        f"bronze.fato_solicitacoes_{ID_FONTE}": df_solicitacoes_fato,
        f"bronze.fato_campos_{ID_FONTE}": df_fato_campos_melt,
        f"bronze.fato_etapas_{ID_FONTE}": df_etapas,
    }

    for nome_tabela, df in tabelas.items():
        sdf = spark.createDataFrame(df)
        sdf.write.mode("overwrite").format("delta").saveAsTable(nome_tabela)
        print(f"✓ {nome_tabela}: {df.shape[0]} linhas")


def gerar_bronze_fato():

    df_solicitacoes, df_etapas = extrair_tabela_acto_gestao(PAYLOAD_PATH, TOKEN)

    df_solicitacoes = df_solicitacoes.rename(
        columns={"n_solicitacao": "id_os"}
    )  # depois de migrar, levar para nb_utils

    df_solicitacoes_fato = gerar_fato_solicitacoes(
        df_solicitacoes, METADADOS_FONTE, COLS_FATO_SOLICITACOES
    )
    df_fato_campos_melt = gerar_fato_campos(
        df_solicitacoes, METADADOS_FONTE, COLS_FATO_SOLICITACOES
    )
    df_etapas = gerar_fato_etapas(df_etapas, METADADOS_FONTE)

    write_bronze(df_solicitacoes_fato, df_fato_campos_melt, df_etapas)


gerar_bronze_fato()